# Main benchmark results

Combines the three runs of `01_main_benchmark.ipynb` and computes the numbers reported in the results: the fold means and standard deviations of Figure 2, the paired permutation tests with Bonferroni correction, and the summary statements.

In [ ]:
import pandas as pd

from config import OUTPUT_DIR

DATASETS = ["PPMI", "NKI", "HBN"]
OUT = OUTPUT_DIR / "main_benchmark"

folds = pd.concat([pd.read_csv(OUT / f"{d}_folds.csv") for d in DATASETS], ignore_index=True)
tests = pd.concat([pd.read_csv(OUT / f"{d}_tests.csv") for d in DATASETS], ignore_index=True)
assert len(tests) == 8, f"expected 8 tests, got {len(tests)}"
folds.to_csv(OUT / "folds.csv", index=False)
tests.to_csv(OUT / "tests.csv", index=False)

## Figure 2

Balanced accuracy for sex and mean absolute error for age and BMI, mean ± standard deviation across the 10 outer folds.

In [ ]:
primary = folds[folds.metric.isin(["balanced_accuracy", "mae"])]
stats = primary.groupby(["dataset", "task", "model"], sort=False)["value"].agg(mean="mean", sd=lambda v: v.std(ddof=0))
cells = stats["mean"].map("{:.3f}".format) + " ± " + stats["sd"].map("{:.3f}".format)
pd.concat({d: cells.loc[d].unstack("task") for d in DATASETS})

## Permutation tests

The un-CNN against the best other model of each panel; `p_bonferroni` is corrected over the 8 tests and `significant` marks corrected $p < 0.05$.

In [ ]:
tests.round(4)

## Summary statements

In [ ]:
means = stats["mean"].unstack("model")
lower_is_better = means.index.get_level_values("task") != "sex"
sign = pd.Series(lower_is_better, index=means.index).map({True: -1, False: 1})
score = means.mul(sign, axis=0)
candidates = score.drop(columns="Dummy")
freesurfer = [m for m in ("FS_Schaefer", "FS_aparc", "FreeSurfer") if m in score]

first = candidates.idxmax(axis=1) == "Untrained_CNN"
beats_fs = score[freesurfer].lt(score["Untrained_CNN"], axis=0).sum(axis=1) == score[freesurfer].notna().sum(axis=1)
above_chance = candidates.gt(score["Dummy"], axis=0) | candidates.isna()
direction = tests.observed_diff * tests.task.map(lambda t: 1 if t == "sex" else -1)

print(f"un-CNN first by mean in {first.sum()} of {len(first)} panels")
print(f"un-CNN significantly better than its strongest competitor in {(tests.significant & (direction > 0)).sum()} of 8 tests, "
      f"significantly worse in {(tests.significant & (direction < 0)).sum()}")
print(f"un-CNN better than every FreeSurfer feature set by mean in {beats_fs.sum()} of {len(beats_fs)} panels")
print(f"every model above the chance baseline in {above_chance.all(axis=1).sum()} of {len(above_chance)} panels")